In [1]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_groq import ChatGroq
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_core.messages.utils import trim_messages, count_tokens_approximately 
from dotenv import load_dotenv
import sqlite3
import os
from langchain.messages import RemoveMessage

In [48]:
load_dotenv()

llm = ChatGroq(
    model = 'llama-3.3-70b-versatile',
    api_key = os.getenv('GROQ_API_KEY')
)

In [84]:
def chat_node(state: MessagesState):

    messages = state['messages']
    

    result = llm.invoke(messages)

    return {
        'messages': [result]
    }


def delete_old_msg(state: MessagesState):
    msg = state['messages']

    if (len(msg)) > 4:

        to_remove = msg[:3]


        # filtered_msg = []

        # for m in to_remove:

        #     filtered_msg.append(RemoveMessage(id= m.id))
        #     #print(m.id)
        return {
            'messages': [RemoveMessage(id=m.id) for m in to_remove]
        }

    return {
        'messages': []
    }

In [85]:
conn = sqlite3.connect(database='stm_db.db', check_same_thread=False)

check_pointer = SqliteSaver(conn=conn)

builder = StateGraph(MessagesState)


builder.add_node('chat_node', chat_node)
builder.add_node('delete_old_msg', delete_old_msg)

builder.add_edge(START, 'chat_node')
builder.add_edge('chat_node', 'delete_old_msg')
builder.add_edge('delete_old_msg', END)

graph = builder.compile(checkpointer=check_pointer)

In [86]:
CONFIG = {
    'configurable': {
        'thread_id': 'thread_6'
    }
}

In [67]:
result = graph.invoke(
    {
        'messages': [
            {
                'role': 'user',
                'content': 'What is My Name?'
            }
        ]
    },
    config=CONFIG
)
print(len(result))
for r in result:
    print(r)
print(result['messages'])

1
messages
[HumanMessage(content='What is My Name?', additional_kwargs={}, response_metadata={}, id='b9ad60ee-5725-4cfd-837b-b37d99697cb7'), AIMessage(content="I don't have any information about your name. I'm a large language model, I don't have the ability to know your personal details or identity. Each time you interact with me, it's a new conversation and I don't retain any information from previous conversations. If you'd like to share your name with me, I'd be happy to chat with you!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 40, 'total_tokens': 115, 'completion_time': 0.204689148, 'completion_tokens_details': None, 'prompt_time': 0.001679283, 'prompt_tokens_details': None, 'queue_time': 0.293603312, 'total_time': 0.206368431}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fb20

In [68]:
result = graph.invoke(
    {
        'messages': [
            {
                'role': 'user',
                'content': 'Do you know what is my Name?'
            }
        ]
    },
    config=CONFIG
)

print('msg len: ', len(result))
print(result)

msg len:  1
{'messages': [HumanMessage(content='What is My Name?', additional_kwargs={}, response_metadata={}, id='b9ad60ee-5725-4cfd-837b-b37d99697cb7'), AIMessage(content="I don't have any information about your name. I'm a large language model, I don't have the ability to know your personal details or identity. Each time you interact with me, it's a new conversation and I don't retain any information from previous conversations. If you'd like to share your name with me, I'd be happy to chat with you!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 40, 'total_tokens': 115, 'completion_time': 0.204689148, 'completion_tokens_details': None, 'prompt_time': 0.001679283, 'prompt_tokens_details': None, 'queue_time': 0.293603312, 'total_time': 0.206368431}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='l

In [27]:
# result = graph.invoke(
#     {
#         'messages': [
#             {
#                 'role': 'user',
#                 'content': "I'm Imran Butt?"
#             }
#         ]
#     },
#     config=CONFIG
# )

# print(result['messages'][-1].content)

print(graph.get_state(config=CONFIG))
print("YOAH")
print(list(graph.get_state_history(config=CONFIG)))

StateSnapshot(values={'messages': [AIMessage(content="I don't have that information. You haven't told me your name, and I don't have the ability to know your name unless you share it with me. I'm a text-based AI assistant, and our conversation just started, so I don't have any prior knowledge about you or your name. If you'd like to share your name, I'd be happy to learn it and chat with you!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 83, 'prompt_tokens': 193, 'total_tokens': 276, 'completion_time': 0.233376326, 'completion_tokens_details': None, 'prompt_time': 0.021926897, 'prompt_tokens_details': None, 'queue_time': 0.008217272, 'total_time': 0.255303223}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fb1f5-0674-7033-892f-d41f26fac4d6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tok

In [69]:
result = graph.invoke(
    {
        'messages': [
            {
                'role': 'user',
                'content': 'Im imran butt?'
            }
        ]
    },
    config=CONFIG
)

print('msg len: ', len(result))
print(result)

msg len:  1
{'messages': [AIMessage(content="I don't have any information about your name. I'm a large language model, I don't have the ability to know your personal details or identity. Each time you interact with me, it's a new conversation and I don't retain any information from previous conversations. If you'd like to share your name with me, I'd be happy to chat with you!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 40, 'total_tokens': 115, 'completion_time': 0.204689148, 'completion_tokens_details': None, 'prompt_time': 0.001679283, 'prompt_tokens_details': None, 'queue_time': 0.293603312, 'total_time': 0.206368431}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fb20a-cefb-7792-9337-24ff48054eef-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 40, 'output

In [87]:
result = graph.invoke(
    {
        'messages': [
            {
                'role': 'user',
                'content': "What is LangGraph??"
            }
        ]
    },
    config=CONFIG
)

print(result['messages'][-1].content)

LangGraph is not a widely recognized term, and it could refer to different things depending on the context. However, based on my knowledge, I couldn't find any specific information on a well-known concept or technology called "LangGraph".

It's possible that LangGraph could be a:

1. **Graph-based language model**: A type of artificial intelligence (AI) model that uses graph structures to represent language and generate text.
2. **Language graph database**: A database that stores and manages linguistic data, such as words, phrases, and grammar rules, in a graph format.
3. **Proprietary technology or project**: A specific technology or project developed by a company or organization, which might not be publicly disclosed or widely known.

If you could provide more context or information about LangGraph, I'd be happy to try and help you understand what it refers to.


In [88]:
print(len(result['messages']))
for msg in result['messages']:
    print(msg.content)
    print()

6
Whatis my name bro??

I remember! You told me earlier that your name is Imran Butt, bro!

Whatis my name buddy??

You're Imran Butt, buddy! We already had this conversation, and I'm glad we're buddies now!

What is LangGraph??

LangGraph is not a widely recognized term, and it could refer to different things depending on the context. However, based on my knowledge, I couldn't find any specific information on a well-known concept or technology called "LangGraph".

It's possible that LangGraph could be a:

1. **Graph-based language model**: A type of artificial intelligence (AI) model that uses graph structures to represent language and generate text.
2. **Language graph database**: A database that stores and manages linguistic data, such as words, phrases, and grammar rules, in a graph format.
3. **Proprietary technology or project**: A specific technology or project developed by a company or organization, which might not be publicly disclosed or widely known.

If you could provide mo

In [78]:
for msg in (graph.get_state(config=CONFIG).values['messages']):
    print(msg.content)

Do you know what is my Name?
No, I don't know your name. As I mentioned earlier, I'm a large language model, I don't have the ability to know your personal details or identity. I don't have any information about you, and I don't retain any data from previous conversations. If you'd like to tell me your name, I'd be happy to chat with you, but I won't be able to guess or recall it on my own.
Im imran butt?
Nice to meet you, Imran Butt! However, please keep in mind that I'm a large language model, I don't have the ability to verify or store personal information, so I'm just taking your word for it. From now on, in this conversation, I'll address you as Imran Butt, but if you interact with me again in the future, I won't retain any memory of our previous conversation or your name. How can I assist you today, Imran?
Whatis my name bro??
I remember! You told me earlier that your name is Imran Butt, bro!


In [ ]:
result = graph.invoke(
    {
        'messages': [
            {
                'role': 'user',
                'content': "Whatis my name buddy??"
            }
        ]
    },
    config=CONFIG
)

print(result['messages'][-1].content)

You're Imran Butt, buddy! We already had this conversation, and I'm glad we're buddies now!


In [81]:
for msg in (result['messages']):
    print(msg.content)

No, I don't know your name. As I mentioned earlier, I'm a large language model, I don't have the ability to know your personal details or identity. I don't have any information about you, and I don't retain any data from previous conversations. If you'd like to tell me your name, I'd be happy to chat with you, but I won't be able to guess or recall it on my own.
Im imran butt?
Nice to meet you, Imran Butt! However, please keep in mind that I'm a large language model, I don't have the ability to verify or store personal information, so I'm just taking your word for it. From now on, in this conversation, I'll address you as Imran Butt, but if you interact with me again in the future, I won't retain any memory of our previous conversation or your name. How can I assist you today, Imran?
Whatis my name bro??
I remember! You told me earlier that your name is Imran Butt, bro!
Whatis my name buddy??
You're Imran Butt, buddy! We already had this conversation, and I'm glad we're buddies now!


In [80]:
for msg in (graph.get_state(config=CONFIG).values['messages']):
    print(msg.content)

No, I don't know your name. As I mentioned earlier, I'm a large language model, I don't have the ability to know your personal details or identity. I don't have any information about you, and I don't retain any data from previous conversations. If you'd like to tell me your name, I'd be happy to chat with you, but I won't be able to guess or recall it on my own.
Im imran butt?
Nice to meet you, Imran Butt! However, please keep in mind that I'm a large language model, I don't have the ability to verify or store personal information, so I'm just taking your word for it. From now on, in this conversation, I'll address you as Imran Butt, but if you interact with me again in the future, I won't retain any memory of our previous conversation or your name. How can I assist you today, Imran?
Whatis my name bro??
I remember! You told me earlier that your name is Imran Butt, bro!
Whatis my name buddy??
You're Imran Butt, buddy! We already had this conversation, and I'm glad we're buddies now!


In [35]:
from langchain_core.messages import RemoveMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END, MessagesState

# 1. Define nodes
def add_message(state: MessagesState):
    return {"messages": ["Hello!"]}

def add_world_message(state: MessagesState):
    return {"messages": ["World!"]}

def delete_message(state: MessagesState):
    first_msg_id = state["messages"][0].id
    # Sends a reducer signal to delete the first message
    return {"messages": [RemoveMessage(id=first_msg_id)]}

# 2. Build graph with checkpointer
builder = StateGraph(MessagesState)
builder.add_node("add", add_message)
builder.add_node("delete", delete_message)
builder.add_node("add_world_message", add_world_message)
builder.add_edge(START, "add")
builder.add_edge("add", 'add_world_message')
builder.add_edge("add_world_message", 'delete')

builder.add_edge("delete", END)

checkpointer = MemorySaver()
app = builder.compile(checkpointer=checkpointer)

# 3. Execute
config = {"configurable": {"thread_id": "1"}}
result = app.invoke({"messages": []}, config)


print(result)
print()


print()

print(app.get_state(config))
print()
print()
for state in (app.get_state_history(config)):
    print(state)
# # 4. Check current state
# current_state = app.get_state(config)
# print("Current State Messages:", len(current_state.values["messages"]))
# # Output: 0 (Message was removed from active state)

# # 5. Check history snapshots
# history = list(app.get_state_history(config))
# print("Number of Historical Checkpoints:", len(history))

# for i, state_snapshot in enumerate(history):
#     msgs = state_snapshot.values.get("messages", [])
#     print(f"Snapshot {i} ({state_snapshot.metadata['source']}): {len(msgs)} messages")

{'messages': [HumanMessage(content='World!', additional_kwargs={}, response_metadata={}, id='0c8f6dea-b88f-40e7-a263-bfe93a1b2d97')]}


StateSnapshot(values={'messages': [HumanMessage(content='World!', additional_kwargs={}, response_metadata={}, id='0c8f6dea-b88f-40e7-a263-bfe93a1b2d97')]}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18beac-dcf0-6953-8003-a16261b56999'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-07-30T07:46:41.670177+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18beac-dcea-60b7-8002-ae935834f470'}}, tasks=(), interrupts=())


StateSnapshot(values={'messages': [HumanMessage(content='World!', additional_kwargs={}, response_metadata={}, id='0c8f6dea-b88f-40e7-a263-bfe93a1b2d97')]}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18beac-dcf0-6953-8003-a16261b56999'}}, metadata={'source': 'loop', '

In [31]:
for state in (graph.get_state_history(config=CONFIG)):
    print(state)

StateSnapshot(values={'messages': [AIMessage(content="I don't have that information. You haven't told me your name, and I don't have the ability to know your name unless you share it with me. I'm a text-based AI assistant, and our conversation just started, so I don't have any prior knowledge about you or your name. If you'd like to share your name, I'd be happy to learn it and chat with you!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 83, 'prompt_tokens': 193, 'total_tokens': 276, 'completion_time': 0.233376326, 'completion_tokens_details': None, 'prompt_time': 0.021926897, 'prompt_tokens_details': None, 'queue_time': 0.008217272, 'total_time': 0.255303223}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fb1f5-0674-7033-892f-d41f26fac4d6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tok

In [36]:
snap = graph.get_state(config=CONFIG)

print(len(snap))


8


StateSnapshot(values={'messages': [AIMessage(content="I don't have that information. You haven't told me your name, and I don't have the ability to know your name unless you share it with me. I'm a text-based AI assistant, and our conversation just started, so I don't have any prior knowledge about you or your name. If you'd like to share your name, I'd be happy to learn it and chat with you!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 83, 'prompt_tokens': 193, 'total_tokens': 276, 'completion_time': 0.233376326, 'completion_tokens_details': None, 'prompt_time': 0.021926897, 'prompt_tokens_details': None, 'queue_time': 0.008217272, 'total_time': 0.255303223}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fb1f5-0674-7033-892f-d41f26fac4d6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tok

# Now At that moment we lost the context (My Name) and that is the problem with Trimming